In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import mplleaflet
import numpy as np
import time
import csv
import rospy
import yaml

from sensor_msgs.msg import NavSatFix

kml_file_path = os.getenv('RUN_LIVE_VIEW_YAML_FILE_PATH')
gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
df = gpd.read_file(kml_file_path, driver='KML')
ax = df.plot()

real_longs = []
real_lats = []
i = 0

odom_longs = []
odom_lats = []
j = 0

def realGpsCallback(msg):
    global ax, df, real_lats, real_longs, i
    
    if i>=50:
        i = 0
        real_lats.append(msg.latitude)
        real_longs.append(msg.longitude)
    else:
        i += 1
        
def odomGpsCallback(msg):
    global ax, df, odom_lats, odom_longs, j
    
    if j>=50:
        j = 0
        odom_lats.append(msg.latitude)
        odom_longs.append(msg.longitude)
    else:
        j += 1

mplleaflet.save_html(fig=ax.figure)
rospy.init_node('map_vis_live')
rospy.Subscriber('/agribot/gnss', NavSatFix, realGpsCallback)
rospy.Subscriber('/agribot/odom_base_navsatfix', NavSatFix, odomGpsCallback)
rate = rospy.Rate(0.2)
ax.hold(True)

while not rospy.is_shutdown():
    if len(real_lats)>0 and len(odom_lats)>0:
        ax.plot(odom_longs, odom_lats, 'b.')
        ax.plot(real_longs, real_lats, 'r.')
        mplleaflet.save_html(fig=ax.figure)
        real_lats = []
        real_longs = []
        odom_lats = []
        odom_longs = []
    rate.sleep()